# GamaX1 (Aetherion) — Colab GPU Training (Bulk / `--data_dir` path)

**Uses the new `bulk_corpus.py` module**, not a single combined `.txt` file. This is the correct path for a large folder of book files (your 2.2GB+ corpus):

- Never loads the whole corpus into RAM as one giant string — it memory-maps an on-disk int32 token cache.
- Prints progress every 500 books while building the cache, so long runs never look frozen.
- Reuses the cache automatically on re-run (checked by file manifest + tokenizer config), so a Colab disconnect doesn't cost you a re-encode.

**Setup order (important):**
1. `Runtime` → `Change runtime type` → Hardware accelerator = **T4 GPU** (or better) → Save.
2. Run cells top to bottom.
3. Put your book `.txt` files in one folder (subfolders OK, they're found recursively) — do NOT pre-combine them into one file. `bulk_corpus.py` handles that internally, book by book.

## 1. Confirm GPU is attached

In [ ]:
!nvidia-smi

If this errors out or shows no GPU, go back to `Runtime` → `Change runtime type`, pick a GPU, then re-run this cell.

## 2. Mount Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os

# PROJECT_ROOT = '/content/drive/MyDrive/Aetherion_GamaX1'
# BOOKS_DIR      = f'{PROJECT_ROOT}/data/books'            # folder of .txt book files (2.2GB+)
# BULK_CACHE_DIR = f'{PROJECT_ROOT}/bulk_cache_multi_v2'   # token cache lives here (on Drive -- must persist)
# CKPT_DIR       = f'{PROJECT_ROOT}/checkpoints_bulk_multi' # model checkpoints live here

# os.makedirs(BOOKS_DIR, exist_ok=True)
# os.makedirs(BULK_CACHE_DIR, exist_ok=True)
# os.makedirs(CKPT_DIR, exist_ok=True)
# print('Books dir  :', BOOKS_DIR)
# print('Bulk cache :', BULK_CACHE_DIR)
# print('Checkpoints:', CKPT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Books dir  : /content/drive/MyDrive/Aetherion_GamaX1/data/books
Bulk cache : /content/drive/MyDrive/Aetherion_GamaX1/bulk_cache_multi
Checkpoints: /content/drive/MyDrive/Aetherion_GamaX1/checkpoints_bulk_multi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/Aetherion_GamaX1'

BOOKS_DIR = f'{PROJECT_ROOT}/data/books'

# Token cache -> Google Drive (FIXED: was previously '/content/...', which is
# Colab's ephemeral local disk. That directory -- including the resumable
# encode_progress.json checkpoint -- is wiped on every runtime disconnect,
# so no matter how good bulk_corpus.py's resume logic is, there was nothing
# left to resume from after any disconnect. Every run silently restarted
# the full encode from file 1. Moving this to Drive fixes that at the root:
# the cache and its checkpoint now survive disconnects like CKPT_DIR always
# has. Named _v2 so this does NOT pick up any old cache dir that may already
# exist on Drive from before this fix (avoids silently trusting a
# possibly-corrupt older cache).
BULK_CACHE_DIR = f'{PROJECT_ROOT}/bulk_cache_multi_v2'

# Checkpoints -> Google Drive (unchanged, already persistent)
CKPT_DIR = f'{PROJECT_ROOT}/checkpoints_bulk_multi'

os.makedirs(BOOKS_DIR, exist_ok=True)
os.makedirs(BULK_CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print('Books dir  :', BOOKS_DIR)
print('Bulk cache :', BULK_CACHE_DIR, '(Drive -- persists across disconnects)')
print('Checkpoints:', CKPT_DIR)


Mounted at /content/drive
Books dir  : /content/drive/MyDrive/Aetherion_GamaX1/data/books
Bulk cache : /content/aetherion_bulk_cache_multi
Checkpoints: /content/drive/MyDrive/Aetherion_GamaX1/checkpoints_bulk_multi


## 3. Get the GamaX1 codebase onto the runtime

**Important:** make sure the version you clone/copy here actually contains `gamax1/bulk_corpus.py` and the `--data_dir` flag in `train.py` — if you pushed those files to GitHub, a plain `git clone`/`git pull` will pick them up. If they only exist locally and were never pushed, Option A (copy from Drive) is safer until you push.

In [ ]:
# Option A -- from GitHub instead (uncomment and edit if you pushed there)
%cd /content
!rm -rf gamax1_project
!git clone https://github.com/mrroy-dev/gamax1.git gamax1_project
%cd /content/gamax1_project
!ls gamax1/

/content
Cloning into 'gamax1_project'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 47 (delta 15), reused 44 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 62.93 KiB | 4.19 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/gamax1_project
bulk_corpus.py	  generate.py  layers.py  tokenizer.py
compare_dense.py  __init__.py  model.py   train.py


In [ ]:
# Sanity check: bulk_corpus.py and the --data_dir flag must both be present,
# AND bulk_corpus.py must be the resumable version (checks for the
# encode_progress checkpoint mechanism). Without this second check, a stale
# clone/copy could silently reintroduce the "never resumes" bug even though
# the file exists and the flag exists.
assert os.path.exists('gamax1/bulk_corpus.py'), 'bulk_corpus.py missing -- wrong code version cloned/copied!'
assert '--data_dir' in open('gamax1/train.py').read(), '--data_dir flag missing from train.py -- wrong code version!'

_bulk_src = open('gamax1/bulk_corpus.py').read()
assert 'encode_progress' in _bulk_src and 'PROGRESS_INTERVAL' in _bulk_src, (
    'bulk_corpus.py does NOT have the resumable-encode fix (no encode_progress '
    'checkpointing found). This is the old version that always restarts the '
    'encode from file 1 after any interruption -- replace it with the fixed '
    'version before continuing.'
)
print('Bulk-training code confirmed present AND resumable.')


Bulk-training code confirmed present.


## 4. Confirm the books are in place

Upload your `.txt` book files into `BOOKS_DIR` (via the Drive web UI, or `rclone`/`gdown`) beforehand -- as separate files, not pre-merged.

In [ ]:
from pathlib import Path

DATA_DIR = Path(BOOKS_DIR).parent  # data/ folder jisme books, wiki, qna sub-folders hain

source_dirs = ["books", "wiki", "qna"]
total_files = 0
total_bytes = 0

for name in source_dirs:
    sub_dir = DATA_DIR / name
    if sub_dir.is_dir():
        files = sorted(sub_dir.rglob('*.txt'))
        file_bytes = sum(p.stat().st_size for p in files)
        total_files += len(files)
        total_bytes += file_bytes
        print(f'{name}: {len(files):,} .txt files, {file_bytes / (1024**3):.2f} GB')
    else:
        print(f'{name}: folder not found, skipping')

print(f'\nTotal: {total_files:,} .txt files, {total_bytes / (1024**3):.2f} GB')
assert total_files, f'No .txt files found under {DATA_DIR} (expected books/wiki/qna sub-folders) -- upload your data first.'

books: 4,614 .txt files, 2.07 GB
wiki: 12,161 .txt files, 0.11 GB
qna: folder not found, skipping

Total: 16,775 .txt files, 2.18 GB


## 5. Install dependencies

In [ ]:
!pip install -q torch --extra-index-url https://download.pytorch.org/whl/cu121
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- check runtime type!')

CUDA available: True
Device: Tesla T4


## 6. Build/reuse the bulk token cache, then train (auto-resume)

First run: this walks every book, encodes it with BPE, and writes the memory-mapped `tokens.int32.bin` cache -- printing progress every 500 books, so you'll see it moving instead of nothing at all. This can take a while for 6.61 the *first* time; every run after that reuses the cache instantly (unless you pass `--rebuild_bulk_cache`).

Training itself auto-resumes from the newest checkpoint in `CKPT_DIR`, same as before -- safe against disconnects.

**Model size:** left at the 1024/16/10/4096 config sized for a ~1.85B-token corpus (~13-14 tokens/param). Adjust below if your actual book collection ends up smaller or larger once the cache reports its real token count.

In [ ]:
import glob

# Naya folder — purana checkpoint yahan nahi hoga, isliye fresh start hoga automatically
CKPT_DIR_256 = f'{CKPT_DIR}_mutli_data'

ckpts = sorted(
    glob.glob(f'{CKPT_DIR_256}/gamax1_step_*.pt'),
    key=lambda p: int(p.rsplit('_', 1)[1].split('.')[0])
)
resume_flag = f'--resume_from "{ckpts[-1]}"' if ckpts else ''
print('Resuming from:', ckpts[-1] if ckpts else '(no checkpoint found -- starting fresh)')

D_MODEL    = 1024
N_HEADS    = 16
N_LAYERS   = 10
N_FEATURES = 4096
BLOCK_SIZE = 256

train_cmd = (
    f'python -m gamax1.train '
    f'--tokenizer bpe '
    f'--data_dir "{DATA_DIR}" ' # Changed from BOOKS_DIR to DATA_DIR to include all subfolders
    f'--bulk_cache_dir "{BULK_CACHE_DIR}" '
    f'--d_model {D_MODEL} '
    f'--n_heads {N_HEADS} '
    f'--n_layers {N_LAYERS} '
    f'--n_features {N_FEATURES} '
    f'--block_size {BLOCK_SIZE} '
    f'--bpe_vocab_size 8000 '
    f'--max_steps 7000 '
    f'--checkpoint_interval 500 '
    f'--out_dir "{CKPT_DIR_256}" '
    f'{resume_flag}'
).strip()

print(train_cmd)

Resuming from: /content/drive/MyDrive/Aetherion_GamaX1/checkpoints_bulk_multi_mutli_data/gamax1_step_5500.pt
python -m gamax1.train --tokenizer bpe --data_dir "/content/drive/MyDrive/Aetherion_GamaX1/data" --bulk_cache_dir "/content/aetherion_bulk_cache_multi" --d_model 1024 --n_heads 16 --n_layers 10 --n_features 4096 --block_size 256 --bpe_vocab_size 8000 --max_steps 7000 --checkpoint_interval 500 --out_dir "/content/drive/MyDrive/Aetherion_GamaX1/checkpoints_bulk_multi_mutli_data" --resume_from "/content/drive/MyDrive/Aetherion_GamaX1/checkpoints_bulk_multi_mutli_data/gamax1_step_5500.pt"


In [ ]:
!{train_cmd}

Using device: cuda
Encoded 500/16,775 files | 70,787,126 tokens | by source: {'books': 70787126, 'wiki': 0} | checkpoint saved
Encoded 1,000/16,775 files | 129,866,921 tokens | by source: {'books': 129866921, 'wiki': 0} | checkpoint saved


## 7. Generate a sample once training is done (or paused)




In [ ]:
!python -m gamax1.generate --ckpt "{CKPT_DIR}/gamax1.pt" --prompt "who is Prince Andrew" --max_new_tokens 500

## Notes / gotchas

- **First-run cache build takes real time** for 2.2GB of books -- watch for the `Encoded N/M files | K tokens | ... | checkpoint saved` progress lines (printed every `PROGRESS_INTERVAL` = 500 files). If you don't see any output for several minutes AND no progress lines, something's actually wrong (wrong code version, or a single giant book file inside `BOOKS_DIR`) -- otherwise, let it run.
- **The cache build IS resumable now, across disconnects too.** `bulk_corpus.py` checkpoints progress to `encode_progress.json` inside `BULK_CACHE_DIR` every 500 files. Since `BULK_CACHE_DIR` now lives on Google Drive (Section 2, fixed), that checkpoint -- and the partially-written `tokens.int32.bin` -- survive a Colab disconnect. Re-running cells 1-6 then cell 6's train cell will pick up encoding right after the last checkpointed file instead of restarting from file 1.
  - Previously this was NOT true: `BULK_CACHE_DIR` pointed at Colab's local `/content/` disk, which is wiped on every disconnect, so the checkpoint was destroyed along with everything else and every run silently restarted from scratch regardless of what the resume code could do. That's fixed now by moving the directory, not by changing `bulk_corpus.py` itself.
  - If the token file on disk is ever shorter than what `encode_progress.json` claims (e.g. a write was interrupted mid-flush), `bulk_corpus.py` detects this and refuses to resume from that checkpoint, forcing a clean re-encode instead of risking a corrupted token stream.
  - If a *completed* cache's `metadata.json` and the actual `tokens.int32.bin` byte size don't match (e.g. Drive sync raced a completed write), `bulk_corpus.py` detects that mismatch too and forces a full re-encode rather than trusting a corrupt cache.
- **If disconnected during training** (cache already built): re-run cells 1-6 -- the cache is reused instantly (or resumed if it wasn't finished) and training resumes from the latest checkpoint in `CKPT_DIR`.
- **`--rebuild_bulk_cache`**: add this flag to `train_cmd` if you ever change the book files or vocab size and need to force a fresh encode, ignoring any existing cache or checkpoint.
- **Drive write throughput**: writing the token cache to Drive instead of local disk is somewhat slower per-byte than local SSD, but this cost is paid once (thanks to resume, at most once total across however many disconnects it takes) and buys actual persistence, which local disk fundamentally cannot. If Drive I/O becomes a real bottleneck for your corpus size, the alternative is keeping `BULK_CACHE_DIR` local for speed and adding an explicit periodic sync of the cache dir to Drive -- more moving parts, only worth it if Drive write speed is measured to actually be a problem.
- Free tier session cap is ~12 hours with no GPU guarantee at peak times; Colab Pro or RunPod/Vast.ai are fallbacks if you hit a wall.
